# Modeling and fitting your velocity field



Our goal in this notebook is to extract the _intrinsic_ rotation curve from your _observed_ MaNGA velocity field.

In the reference frame of the galaxy (i.e, if we could somehow float above it and look down on what's going on), stars and gas are rotating about its center with a speed that depends on their distance from the center.  For the most part, the orbits are confined to a plane, the galaxy "disk."  The relation between radius and speed is the "rotation curve" and tells us about the mass enclosed.

But our data is not in the reference frame of the galaxy.  We say that it's in the reference frame of the sky (or the observer).  Unless you have chosen a perfectly edge-on disk galaxy, you have to account for the _inclination_ of the galaxy disk with respect to our line of sight.

This is a **very common** problem in science.  We observe something related to, but not exactly a measure of the intrinsic thing we care about.  The solution is to make a "model" of the intrinsic properties we care about.  Then, to fit the model to the data.  The model must have some flexibility and must account for errors in our measurements.  Finally, the model will never be perfect and folks will argue about whether the model is "good enough" to give you an unbiased view of the result (e.g., how much mass is enclosed?) that you're after...

## Our disk galaxy velocity model

We start with a simple 2-parameter model for the rotation curve that is known to work pretty well:

$$ V(r) = V_{rot} = V_{max} \tanh(r/h) $$

$V(r)$ is the model velocity (which depends on radius), while the radius from the galaxy center is $r$. 

The $V_{max}$ parameter defines the (maximum) rotational velocity at ends of galaxy where the model rotation curve  flattens.  It normalizes the overall amplitude of velocities in the rotation curve.

Meanwhile $h$ is a simple scaling parameter (in the same units as radius) that determines how quickly the model rotation curve rises in the center.  

### Let's plot this velocity model

* Define $r$ to be an array that varies from -5 kpc (kiloparsec, unit of distance) to +5 kpc
* Set $V_{max}$ to 150 km/s
* Set $h = 0.5$ kpc

Plese plot this curve, and then overplot 2-3 additional versions with some different values of $V_{max}$ and $h$ to illuminate the effect of changing these parameters.

In [ ]:
# Imports and preliminaries
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
#%matplotlib notebook
from astropy.io import fits
import astropy

# For auto reload of modules that have changed
%load_ext autoreload
%autoreload 2

In [ ]:
# In this cell, plot the velocity model

## Projecting the velocity model onto the sky

We now need to consider how our model for the rotating disk can be projected into a 2D velocity field as we observe on-sky.  As shown by the figure below, we need to translate between the galaxy-frame coordinates (left panel) and the observer-frame coordinates (right panel).

<div>
<img src="rotCurve_coordinate_systems.jpg" width="700"/>
</div>

In the galaxy frame (panel a) we are hovering above the galaxy, looking down at a polar coordinate system centered on the galaxy's center.  The radius, $r$, is measured within the disk and is the same $r$ as in our rotation curve model.  Likewise, $V(r)$ is the rotational velocity in the model that we plotted above.  We also have cartesian coordinates in the disk plane, $x_d$ and $y_d$.  The $\theta$ coordinate is an angle measured with respect to the axis of inclination (which measures the "tilt" of the disk with repect to our line of sight).  The rotation curve model _in the frame of the galaxy_ does not depend on $\theta$... in other words, the model assumes that all disk material is (on average, at least) rotating on circular orbits.

That inclination axis, labeled $i$, in panel (a), conveys the idea that from our vantage point as the observer, the disk will be randomly oriented in space with some possible tilt, or inclination.  In panel (b), we see the projected coordinate system of our observations of a tilted disk as projected "on-sky".  Astronomical data usually uses a "celestial coordinate system" that is a lot like the lattitude-longitude coordinate system on earth.  But instead of looking down, as we do when we think about map coordinates on earth, for celestial coordinates, we are on the inside looking up.  That's why, on the sky, North points up and East points _left_.  The pixel positions in the MaNGA data cubes are equivalent to $(x_s, y_s)$ positions in panel (b).  Note that the on-sky angle of the inclination axis is called the "position angle" (PA).  

We also see in panel (b) that the disk is "foreshortened" according to how much it is inclined.  In fact, it might occur to you that the ratio of the observed width of the minor axis, $b$, compared to the major axis, $a$, can tell us what the inclination angle, $i$, actually is.  Finally, thanks to the inclination, material in the disk is now moving either towards or away from us.  If we can figure out the inclination, we can now use redshift (Doppler shift) to infer the rotational velocity (!). 

With these coordinates defined, the 2D map of the observed line-of-sight velocity is:

$$V(r)_{\rm LOS} = V(r)_{\rm rot} \cos\theta \sin i $$

As we'll see, the hard part is mapping $(x_s, y_s)$ to $(r, \theta)$, so we can use the equation above.

### Generate a synthetic 2D velocity field

Luckily, we've done this part for you!  What you need to do is read through the `velocity_field` python "class" in `mangaRotCurves.py` and understand the steps involved.  A class, by the way, is a self-contained collection of functions and properties, a convenient way of holding together a set of variables and methods in a single variable that are all related to one another.  You've been using classes without realizing it perhaps.  The pandas `DataFrame` and numpy's `array` are classes for example.

In [ ]:
# Import the velocity_field class
# We assume that mangaRotCurves.py is in the working, notebook directory
from mangaRotCurves import velocity_field as vf

In [ ]:
# Generate the basis of a fake velocity field assuming a 64x64 pixel map of velocities.

# Get the pixel coordinates and create a coordinate grid for x and y
pix = np.arange(64).astype(float)
x, y = np.meshgrid(pix[::-1], pix)
# Note that we reversed the order of the x axis; this is a sublety that
# has to do with the MaNGA maps being sky-right.

# Set all the errors to be the same
ve = np.full_like(x, 5.0)
# Instantiate the velocity field without any velocities at first,
# but then use the internal `sample` function to construct the synthetic data
VF = vf(x, y, None, ve)

In [ ]:
# Set initial parameters:  
#
# You need to define the parameter array `init_par`, which is supplied to the `sample` function.
# You will want to experiment with these parameters to see what impact they have.
# But here are some suggestions:
#   - galaxy center: Look at how (xs, ys) is defined in `polar_coordinates`.  
#      To get the center to be (0,0) the first two parameters need to be negative
#   - position angle (PA): Up to you, but maybe try something like 45 deg 
#   - asymptotic rotation speed (Vmax): Should be around 100-200 km/s
#   - rotation scale (h): defined in pixels, should be a decent fraction of the cube (e.g. 10 pixels)
#   - systemic or bulk velocity: 0 km/s for now

#init_par = ... # Fill this in

In [ ]:
# This call to np.random.normal both samples the 2D velocity field and adds noise to it.
VF.v = np.random.normal(VF.sample(init_par), ve)

In [ ]:
# Plot your synthetic 2D velocity field
#   Use imshow with origin='lower', interpolation='nearest', cmap='RdBu_r' and maybe experiment with extent=

### Synthetic velocity field: How did it go?

Did you get something like this?
<div>
<img src="example_velocity_field.jpg" width="300"/>
</div>

This is a realization of our velocity model as we might observe it in real MaNGA data.  We added noise, which makes it look like data, but notice that the velocity field extends as far as we want.  We haven't included effects in this model like the observing field of view or the decrease in signal-to-noise in the faint outskirts of typical galaxies.

BUT, the power here is that we can now compare realizations of this model to real, observed MaNGA velocity fields.  By adjusting the model to "fit" the data, we can recover the best parameters of the rotation curve model that describe the data.

## Test the Fitter: Fit the synthetic data

Let's see how well we can use our model to fit the synthetic velocity field we just made.  This is a bit circular, because we used the model to make the velocity field, but we did add some noise, so we wouldn't expect perfect recovery.  In general, this is a good initial test of a model fitting approach.

The machinery for performing the fit is written up for you in the `velocity_field` class.  And the fit itself is handled by a function called `least_squares` from the `scipy` package.  Nice, right?

In [ ]:
# Import the `least_squares` function
from scipy.optimize import least_squares

In [ ]:
# Set the guess parameters for the velocity field model,
# slightly offset from the true parameters.
# guess_par = ... # you fill this in

In [ ]:
# Because the velocity-field fitting has many parameters, and
# because some of those parameters have natural limits (e.g.,
# 0 < inclination < 90), we impose lower and upper boundaries
# on the allowed parameter values.  +/- np.inf means there is
# no limit.
lower_bounds = np.array([-64, -64,   0,  5,   1, 0.1, -np.inf])
upper_bounds = np.array([  0,   0, 360, 85, 500, 100,  np.inf])

# Run the fit, get the errors in the parameters
result = least_squares(VF.chi, guess_par, bounds=(lower_bounds, upper_bounds))
covar = np.linalg.inv(np.dot(result.jac.T,result.jac))
par_err = np.sqrt(np.diagonal(covar)) # This gives us the errors on fit parameters

In [ ]:
# For each parameter, print out a nicely formatted table of
#   'Parameter', 'Input Value', 'Best Fit Value', 'Error on Best Fit Value'

## Plot the rotation curve implied by the best fit

Okay, you should have fitting parameters that look close (but not exactly) like your input model parameters.

Let's imagine this was real galaxy data and plot the implied rotation curve from the model fit!

In [ ]:
# Use the best-fitting result to get the disk-plane polar coordinates
r, theta = VF.polar_coordinates(result.x[:4])

In [ ]:
# Deproject the rotation velocities; i.e., use some algebra to invert the Vlos equation
# above to solve for Vrot given Vlos
# vrot = ...  # Fill this in

In [ ]:
# The deprojection near the minor axes (where theta is +/- 90) is very large, so
# let's only plot data near the major axis.  Let's select data within a +/- 30 deg
# wedge around the major axis in indx
# wedge = 30
# indx = ... # fill this in, this is a multi-conditional boolean indx on theta

In [ ]:
# Plot the result

# First plot the data (e.g., using plt.scatter() on r[indx] and vrot[indx])
# Your code here

# Then overplot the best-fitting model
#   You'll need to generate an array of radii input values
#   Then use VF.rotation_curve() to return the model velocity as a function of radius,
#        given the bestfit parameters in result.X

You should get get something like this...
<div>
<img src="example_rotation_curve_fit.jpg" width="300"/>
</div>